In [51]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"


In [52]:
# === 0. Setup ===
import os
import math
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict
from datetime import timedelta

# Optional (for torch Dataset skeleton – 학습 단계에서 유용)
try:
    import torch
    from torch.utils.data import Dataset
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False
    print("[Info] PyTorch not installed. You can still run up to fold-splitting.")

@dataclass
class Config:
    lookback: int = 28   # L
    horizon: int  = 7    # H
    patch_len: int = 7   # for PatchTST later
    stride: int    = 1   # for PatchTST later
    n_splits: int  = 5   # K-fold
    embargo_days: int = 35  # purge gap around validation (≈ lookback + horizon)
    seed: int = 42

CFG = Config()
np.random.seed(CFG.seed)

print(CFG)


Config(lookback=28, horizon=7, patch_len=7, stride=1, n_splits=5, embargo_days=35, seed=42)


In [53]:
# === 1. Load & sort ===
# 경로는 ipynb 기준으로 맞춰주세요.
TRAIN_PATH = "./data_filtering/filtered/train.csv"  # ex) "./dataset/train/train.csv"

df = pd.read_csv(TRAIN_PATH)

# Basic parsing
df['date'] = pd.to_datetime(df['date'])
# 안전장치: store_menu 없으면 생성
if 'store_menu' not in df.columns:
    df['store_menu'] = df['store'].astype(str) + "_" + df['menu'].astype(str)

# 정렬 & 타입 정리
df = df.sort_values(['store_menu', 'date']).reset_index(drop=True)

print("Rows:", len(df))
print("Unique store_menu:", df['store_menu'].nunique())
print(df.head(3))


Rows: 102676
Unique store_menu: 193
   date_ordinal       date          store_menu       store     menu  sales
0        738521 2023-01-01  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ  1인 수저세트      0
1        738522 2023-01-02  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ  1인 수저세트      0
2        738523 2023-01-03  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ  1인 수저세트      0


In [54]:
# === 2. Feature engineering (calendar only, rule-safe) ===
def add_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    f = frame.copy()
    f['dow'] = f['date'].dt.weekday           # 0..6
    f['dom'] = f['date'].dt.day               # 1..31
    f['month'] = f['date'].dt.month           # 1..12
    f['is_weekend'] = (f['dow'] >= 5).astype(int)

    # Sine/Cos encoding for weekly seasonality
    f['dow_sin'] = np.sin(2 * np.pi * f['dow'] / 7)
    f['dow_cos'] = np.cos(2 * np.pi * f['dow'] / 7)
    return f

df_feat = add_calendar_features(df)

# 사용할 입력 피처 목록 (sales + calendar)
FEATURE_COLS = ['sales', 'dow', 'dom', 'month', 'is_weekend', 'dow_sin', 'dow_cos']
TARGET_COL = 'sales'

print("Feature columns:", FEATURE_COLS)
df_feat.head(3)


Feature columns: ['sales', 'dow', 'dom', 'month', 'is_weekend', 'dow_sin', 'dow_cos']


,date_ordinal,date,store_menu,store,menu,sales,dow,dom,month,is_weekend,dow_sin,dow_cos
0,738521,2023-01-01,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0,6,1,1,1,-0.781831,0.62349
1,738522,2023-01-02,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0,0,2,1,0,0.000000,1.00000
2,738523,2023-01-03,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0,1,3,1,0,0.781831,0.62349


In [55]:
# === 3. Sliding windows (28 -> 7) indexing ===
def build_samples_index(
    frame: pd.DataFrame,
    lookback: int,
    horizon: int,
    feature_cols: List[str],
    target_col: str = 'sales'
) -> pd.DataFrame:
    """
    Returns a 'samples' DataFrame where each row = one (X:lookback, Y:horizon) sample.
    Columns:
      - sample_id (int)
      - store_menu (str)
      - input_start_date, input_end_date, target_start_date, target_end_date (datetime)
      - input_start_idx, target_start_idx (int, relative within each group)
    """
    rows = []
    sample_id = 0

    for sm, g in frame.groupby('store_menu', sort=False):
        g = g.sort_values('date').reset_index(drop=True)
        n = len(g)
        # last valid target start index (inclusive)
        last_t = n - horizon
        # first valid target start index (input must fully exist)
        first_t = lookback
        for t in range(first_t, last_t + 1):
            inp_start = t - lookback
            inp_end   = t - 1
            tgt_start = t
            tgt_end   = t + horizon - 1

            rows.append({
                'sample_id': sample_id,
                'store_menu': sm,
                'input_start_idx': inp_start,
                'target_start_idx': tgt_start,
                'input_start_date': g.loc[inp_start, 'date'],
                'input_end_date':   g.loc[inp_end,   'date'],
                'target_start_date':g.loc[tgt_start, 'date'],
                'target_end_date':  g.loc[tgt_end,   'date'],
            })
            sample_id += 1

    samples = pd.DataFrame(rows).sort_values(['store_menu','target_start_date']).reset_index(drop=True)
    return samples

samples = build_samples_index(df_feat, CFG.lookback, CFG.horizon, FEATURE_COLS, TARGET_COL)
print("Total samples:", len(samples))
samples.head(3)


# Optional: PyTorch dataset skeleton for later training
if TORCH_AVAILABLE:
    class SalesWindowDataset(Dataset):
        """
        Lazily slices windows from the base frame using 'samples' index.
        Assumes the base frame has been sorted by ['store_menu', 'date'].
        """
        def __init__(self, base_df: pd.DataFrame, samples_df: pd.DataFrame,
                     feature_cols: List[str], target_col: str,
                     lookback: int, horizon: int):
            self.base = base_df
            self.samples = samples_df
            self.feat_cols = feature_cols
            self.tgt_col = target_col
            self.L = lookback
            self.H = horizon

            # Pre-build group offsets for fast slicing
            self.group_index: Dict[str, pd.DataFrame] = {
                sm: g.reset_index(drop=True) for sm, g in self.base.groupby('store_menu', sort=False)
            }

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            s = self.samples.iloc[idx]
            g = self.group_index[s['store_menu']]

            inp_start = int(s['input_start_idx'])
            inp_end   = inp_start + self.L
            tgt_start = int(s['target_start_idx'])
            tgt_end   = tgt_start + self.H

            X = g.loc[inp_start:inp_end-1, self.feat_cols].to_numpy(dtype=np.float32)  # (L, C)
            y = g.loc[tgt_start:tgt_end-1, self.tgt_col].to_numpy(dtype=np.float32)    # (H,)

            # Safety clamp (for later): predictions should be >= 0; labels here as-is.
            return torch.from_numpy(X), torch.from_numpy(y)



Traceback (most recent call last):
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/multiprocessing/queues.py", line 250, in _feed
    send_bytes(obj)
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/multiprocessing/connection.py", line 200, in send_bytes
    self._send_bytes(m[offset:offset + size])
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/multiprocessing/connection.py", line 411, in _send_bytes
    self._send(header + buf)
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/multiprocessing/connection.py", line 368, in _send
    n = write(self._handle, buf)
OSError: [Errno 9] Bad file descriptor
Traceback (most recent call last):
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/multiprocessing/queues.py", line 239, in _feed
    reader_close()
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.10/multiprocessing/connection.py", line 177, in close
    self._close()
  File "/home/wonjun/.conda/envs/wonjun_base/lib/python3.

Total samples: 96114


In [56]:
# === 4. Time-aware K-fold with embargo (purged) ===
# === REPLACE: Time-aware K-fold with warm-up & safety ===
def build_time_kfold_splits_safe(
    samples: pd.DataFrame,
    n_splits: int,
    lookback: int,
    horizon: int,
    embargo_days: int,
    verbose: bool = True,
):
    s = samples.sort_values('target_start_date').reset_index(drop=True)

    # 1) 고유 검증 후보 날짜(실제 샘플이 존재하는 날짜만)
    uniq_dates = pd.Series(s['target_start_date'].unique()).sort_values().to_list()

    # 2) warm-up 이후만 검증으로 사용
    warmup = lookback + horizon + embargo_days   # 최소 70일 권장
    earliest_val_date = pd.Timestamp(uniq_dates[0]) + pd.Timedelta(days=warmup)
    val_date_candidates = [d for d in uniq_dates if d >= earliest_val_date]

    if len(val_date_candidates) < n_splits:
        if verbose:
            print(f"[Warn] 검증 후보 날짜가 {len(val_date_candidates)}일 뿐입니다. "
                  f"요청한 n_splits={n_splits} → {len(val_date_candidates)}로 축소.")
        n_splits = max(1, len(val_date_candidates))

    bins = np.array_split(np.array(val_date_candidates), n_splits)

    folds = []
    for k, val_dates in enumerate(bins, start=1):
        if len(val_dates) == 0:
            if verbose: print(f"[Skip] Fold {k}: 빈 검증 구간")
            continue

        val_start = pd.Timestamp(val_dates[0])
        val_end   = pd.Timestamp(val_dates[-1])
        val_mask  = s['target_start_date'].isin(val_dates)
        val_idx   = s.index[val_mask].to_numpy()

        # 학습은 검증 시작일 - embargo 이전의 것만
        cutoff = val_start - pd.Timedelta(days=embargo_days)
        train_mask = s['target_end_date'] < cutoff
        train_idx  = s.index[train_mask].to_numpy()

        if len(train_idx) == 0 or len(val_idx) == 0:
            if verbose:
                print(f"[Skip] Fold {k}: train={len(train_idx)}, val={len(val_idx)} (warm-up/embargo 과도) → 건너뜀")
            continue

        folds.append((train_idx, val_idx))
        if verbose:
            print(f"[Fold {len(folds)}/{n_splits}] "
                  f"Val {val_start.date()}→{val_end.date()} | "
                  f"train_size={len(train_idx):,}, val_size={len(val_idx):,}")

    # 안전장치: 남은 fold가 1개 이하라면 embargo를 완화해서 다시 시도
    if len(folds) <= 1:
        if verbose:
            print("[Info] 유효 fold가 너무 적습니다. embargo를 완화(예: lookback)하여 재시도합니다.")
        relaxed_embargo = max(lookback, embargo_days // 2)  # 최소 lookback
        return build_time_kfold_splits_safe(
            samples, n_splits, lookback, horizon, relaxed_embargo, verbose
        )

    return folds


folds = build_time_kfold_splits_safe(
    samples=samples,
    n_splits=CFG.n_splits,
    lookback=CFG.lookback,
    horizon=CFG.horizon,
    embargo_days=CFG.embargo_days,   # 35
    verbose=True
)


# (Optional) torch Dataset 예시 바인딩
if TORCH_AVAILABLE:
    full_dataset = SalesWindowDataset(
        base_df=df_feat,
        samples_df=samples,
        feature_cols=FEATURE_COLS,
        target_col=TARGET_COL,
        lookback=CFG.lookback,
        horizon=CFG.horizon
    )

    # Example: first fold indices
    tr_idx, va_idx = folds[0]
    print(f"First fold -> train:{len(tr_idx)}, val:{len(va_idx)}")


[Fold 1/5] Val 2023-04-09→2023-07-03 | train_size=5,597, val_size=16,598
[Fold 2/5] Val 2023-07-04→2023-09-27 | train_size=22,195, val_size=16,598
[Fold 3/5] Val 2023-09-28→2023-12-22 | train_size=38,793, val_size=16,598
[Fold 4/5] Val 2023-12-23→2024-03-16 | train_size=55,391, val_size=16,405
[Fold 5/5] Val 2024-03-17→2024-06-09 | train_size=71,796, val_size=16,405
First fold -> train:5597, val:16598


In [57]:
# === 6. PatchTST (minimal) + RevIN — PATCHED ===
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class RevIN(nn.Module):
    """Per-sample, per-channel normalization with reversible denorm (stabilized)."""
    def __init__(self, eps: float = 1e-5, min_std: float = 1.0):
        super().__init__()
        self.eps = eps
        self.min_std = min_std  # <--- 추가: 너무 작은 분산 방지

    def forward(self, x, sales_ch: int = 0, stats=None, mode='norm'):
        # x: (B, T, C)
        if mode == 'norm':
            mu = x.mean(dim=1, keepdim=True)                       # (B,1,C)
            sigma = x.std(dim=1, keepdim=True) + self.eps          # (B,1,C)
            sigma = torch.clamp(sigma, min=self.min_std)           # <--- 추가
            x_n = (x - mu) / sigma
            mu_s = mu[:, :, sales_ch]                              # (B,1)
            sg_s = sigma[:, :, sales_ch]                           # (B,1)
            return x_n, (mu_s, sg_s)
        elif mode == 'denorm':
            mu_s, sg_s = stats                                     # (B,1), (B,1)
            y = x                                                  # (B,H)
            return y * sg_s + mu_s
        else:
            raise ValueError("mode must be 'norm' or 'denorm'")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.cos(pos * div)
        pe[:, 1::2] = torch.sin(pos * div)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        P = x.size(1)
        return x + self.pe[:, :P, :]

class PatchTSTMini(nn.Module):
    def __init__(
        self,
        lookback: int,
        horizon: int,
        c_in: int,
        d_model: int = 128,
        n_heads: int = 4,
        depth: int = 3,
        patch_len: int = 7,
        stride: int = 1,
        dropout: float = 0.1,
        sales_ch: int = 0
    ):
        super().__init__()
        self.L = lookback
        self.H = horizon
        self.C = c_in
        self.patch_len = patch_len
        self.stride = stride
        self.sales_ch = sales_ch

        self.P = 1 + (self.L - self.patch_len) // self.stride

        self.embed = nn.Linear(self.patch_len * self.C, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.posenc = PositionalEncoding(d_model=d_model, max_len=self.P)

        # HEAD
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, self.H)
        )

        # RevIN with min_std clamp
        self.revin = RevIN(eps=1e-5, min_std=1.0)

    def patchify(self, x):
        # x: (B, L, C) -> (B, P, patch_len*C)
        B, L, C = x.shape
        patches = []
        for start in range(0, L - self.patch_len + 1, self.stride):
            end = start + self.patch_len
            p = x[:, start:end, :].reshape(B, -1)
            patches.append(p)
        return torch.stack(patches, dim=1)

    def forward(self, x, y=None):
        # 1) RevIN
        x_n, stats = self.revin(x, sales_ch=self.sales_ch, mode='norm')

        # 2) Patch + Encoder
        patches = self.patchify(x_n)           # (B,P,patch_len*C)
        z = self.embed(patches)                # (B,P,d)
        z = self.posenc(z)
        z = self.encoder(z)                    # (B,P,d)

        # 3) **Last-token pooling** (mean pooling 제거)
        z = z[:, -1, :]                        # <--- 핵심 수정

        # 4) Head
        y_hat_n = self.head(z)                 # (B,H)
        return y_hat_n, stats


Device: cuda


In [58]:
# === 7. Metrics, Train/Eval, K-fold training — PATCHED ===
from copy import deepcopy
from time import time

def smape_ignore_zero(y_true, y_pred, eps=1e-6):
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)
    mask = (y_true != 0)
    if mask.sum() == 0:
        return 0.0
    yt = y_true[mask]
    yp = y_pred[mask]
    return (100.0 * torch.mean(2.0 * torch.abs(yp - yt) / (torch.abs(yt) + torch.abs(yp) + eps))).item()

def train_one_fold(
    fold_id: int,
    model_cfg: dict,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    base_df: pd.DataFrame,
    samples_df: pd.DataFrame,
    feature_cols: List[str],
    target_col: str,
    batch_size: int = 256,
    max_epochs: int = 60,
    lr: float = 5e-4,            # 살짝 낮춤 (denorm MAE 안정화)
    patience: int = 10,
    mask_zero_in_loss: bool = False   # 필요 시 True
):
    ds_full = SalesWindowDataset(
        base_df=base_df, samples_df=samples_df,
        feature_cols=feature_cols, target_col=target_col,
        lookback=CFG.lookback, horizon=CFG.horizon
    )
    ds_tr = Subset(ds_full, train_idx)
    ds_va = Subset(ds_full, val_idx)

    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = PatchTSTMini(
        lookback=CFG.lookback, horizon=CFG.horizon, c_in=len(feature_cols),
        d_model=model_cfg.get('d_model', 128), n_heads=model_cfg.get('n_heads', 4),
        depth=model_cfg.get('depth', 3), patch_len=CFG.patch_len, stride=CFG.stride,
        dropout=model_cfg.get('dropout', 0.1), sales_ch=0
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    best_model = None
    best_smape = float('inf')
    wait = 0

    L1 = nn.L1Loss(reduction='none')  # denorm MAE에 사용

    for epoch in range(1, max_epochs+1):
        # ---- Train ----
        model.train()
        tr_loss = 0.0
        for X, y in dl_tr:
            X = X.to(device)  # (B,L,C)
            y = y.to(device)  # (B,H)

            opt.zero_grad()

            # 모델 출력(정규화 공간) -> denorm 후 원 스케일 MAE
            y_hat_n, stats = model(X)
            y_hat = model.revin(y_hat_n, stats=stats, mode='denorm')  # (B,H)
            y_hat = torch.clamp(y_hat, min=0.0)

            if mask_zero_in_loss:
                mask = (y != 0).float()
                loss = (L1(y_hat, y) * mask).sum() / (mask.sum() + 1e-6)
            else:
                loss = L1(y_hat, y).mean()

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            tr_loss += loss.item() * X.size(0)

        tr_loss /= len(ds_tr)
        sch.step()

        # ---- Validate ----
        model.eval()
        with torch.no_grad():
            all_y, all_pred = [], []
            for Xv, yv in dl_va:
                Xv, yv = Xv.to(device), yv.to(device)
                y_hat_n, stats = model(Xv)
                y_hat = model.revin(y_hat_n, stats=stats, mode='denorm')
                y_hat = torch.clamp(y_hat, min=0.0)
                all_y.append(yv)
                all_pred.append(y_hat)

            Y = torch.cat(all_y, dim=0)
            P = torch.cat(all_pred, dim=0)
            val_smape = smape_ignore_zero(Y, P)

        print(f"[Fold {fold_id}] Epoch {epoch:03d} | train_loss={tr_loss:.5f} | val_sMAPE={val_smape:.3f}")

        # Early stopping on SMAPE
        if val_smape + 1e-6 < best_smape:
            best_smape = val_smape
            best_model = deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"[Fold {fold_id}] Early stop. Best sMAPE={best_smape:.3f}")
                break

    ckpt_path = f"./patchtst_fold{fold_id}.pt"
    torch.save(best_model, ckpt_path)
    print(f"[Fold {fold_id}] Saved best model -> {ckpt_path}")
    return best_smape, ckpt_path



# === Run K-fold training ===
model_cfg = dict(d_model=128, n_heads=4, depth=3, dropout=0.1)
fold_ckpts = []
fold_scores = []

for k, (tr_idx, va_idx) in enumerate(folds, start=1):
    s, p = train_one_fold(
        fold_id=k,
        model_cfg=model_cfg,
        train_idx=tr_idx,
        val_idx=va_idx,
        base_df=df_feat,
        samples_df=samples,
        feature_cols=FEATURE_COLS,
        target_col=TARGET_COL,
        batch_size=256,
        max_epochs=40,
        lr=1e-3,
        patience=6,
    )
    fold_scores.append(s)
    fold_ckpts.append(p)

print("Fold sMAPE:", fold_scores)
print("Avg sMAPE:", sum(fold_scores)/len(fold_scores))

[Fold 1] Epoch 001 | train_loss=4.66580 | val_sMAPE=126.930
[Fold 1] Epoch 002 | train_loss=4.33234 | val_sMAPE=125.577
[Fold 1] Epoch 003 | train_loss=4.22853 | val_sMAPE=119.855
[Fold 1] Epoch 004 | train_loss=4.13045 | val_sMAPE=121.203
[Fold 1] Epoch 005 | train_loss=4.04202 | val_sMAPE=107.534
[Fold 1] Epoch 006 | train_loss=3.93736 | val_sMAPE=108.677
[Fold 1] Epoch 007 | train_loss=3.85148 | val_sMAPE=116.475
[Fold 1] Epoch 008 | train_loss=3.76688 | val_sMAPE=117.118
[Fold 1] Epoch 009 | train_loss=3.65504 | val_sMAPE=103.123
[Fold 1] Epoch 010 | train_loss=3.56908 | val_sMAPE=107.111
[Fold 1] Epoch 011 | train_loss=3.50808 | val_sMAPE=105.685
[Fold 1] Epoch 012 | train_loss=3.41067 | val_sMAPE=107.943
[Fold 1] Epoch 013 | train_loss=3.39918 | val_sMAPE=112.316
[Fold 1] Epoch 014 | train_loss=3.33707 | val_sMAPE=113.296
[Fold 1] Epoch 015 | train_loss=3.25083 | val_sMAPE=113.732
[Fold 1] Early stop. Best sMAPE=103.123
[Fold 1] Saved best model -> ./patchtst_fold1.pt
[Fold 2] Ep

In [59]:
# === 8. Inference for TEST files (28->7), fold ensemble ===
import glob

def build_input_tensor_from_block(block_df: pd.DataFrame, feature_cols: List[str]) -> torch.Tensor:
    """ block_df: one store_menu, 28 rows sorted by date """
    g = block_df.sort_values('date')
    X = g[feature_cols].to_numpy(dtype=np.float32)  # (28,C)
    return torch.from_numpy(X).unsqueeze(0)  # (1,28,C)

@torch.no_grad()
def predict_7days_for_testfile(
    test_path: str,
    ckpt_paths: List[str],
    feature_cols: List[str],
    save_path: str
):
    test = pd.read_csv(test_path)
    test['date'] = pd.to_datetime(test['date'])
    if 'store_menu' not in test.columns:
        test['store_menu'] = test['store'].astype(str) + "_" + test['menu'].astype(str)
    test = test.sort_values(['store_menu','date']).reset_index(drop=True)

    # 같은 파생 피처 함수 재사용
    test_feat = add_calendar_features(test)

    # Target 날짜 7일 생성
    last_date = test_feat['date'].max()
    target_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=CFG.horizon, freq='D')

    # 준비: fold 모델들 로드
    models = []
    for ck in ckpt_paths:
        m = PatchTSTMini(
            lookback=CFG.lookback,
            horizon=CFG.horizon,
            c_in=len(feature_cols),
            d_model=model_cfg.get('d_model',128),
            n_heads=model_cfg.get('n_heads',4),
            depth=model_cfg.get('depth',3),
            patch_len=CFG.patch_len,
            stride=CFG.stride,
            dropout=model_cfg.get('dropout',0.1),
            sales_ch=0
        ).to(device)
        sd = torch.load(ck, map_location=device)
        m.load_state_dict(sd)
        m.eval()
        models.append(m)

    # store_menu별 예측
    preds = {}
    for sm, g in test_feat.groupby('store_menu', sort=False):
        assert len(g) >= CFG.lookback, f"{sm}: need at least {CFG.lookback} rows"
        # 마지막 28일만 사용 (룰 준수)
        g_last = g.tail(CFG.lookback)
        X = build_input_tensor_from_block(g_last, feature_cols).to(device)  # (1,28,C)

        # fold 앙상블
        fold_outs = []
        for m in models:
            y_hat_n, stats = m(X)                      # normalized pred
            y_hat = m.revin(y_hat_n, stats=stats, mode='denorm')  # (1,7)
            y_hat = torch.clamp(y_hat, min=0.0)
            fold_outs.append(y_hat)

        y_mean = torch.mean(torch.stack(fold_outs, dim=0), dim=0).squeeze(0).cpu().numpy()  # (7,)
        preds[sm] = y_mean

    # 제출 포맷: rows=7일, cols=store_menu
    sm_list = list(test_feat['store_menu'].drop_duplicates())
    sub = pd.DataFrame(index=target_dates, columns=sm_list, dtype=float)
    for sm in sm_list:
        sub[sm] = preds[sm]

    # CSV 저장
    sub.index.name = 'date'
    sub.to_csv(save_path)
    print(f"[Saved] {save_path} | shape={sub.shape}")
    return sub

# TEST 파일들 일괄 예측
test_files = sorted(glob.glob("./data_filtering/filtered/TEST_0*.csv"))
print("Found test files:", test_files)

for tp in test_files:
    tag = os.path.splitext(os.path.basename(tp))[0]   # e.g., TEST_00
    out_csv = f"./submission_{tag}.csv"
    _ = predict_7days_for_testfile(
        test_path=tp,
        ckpt_paths=fold_ckpts,       # fold 앙상블
        feature_cols=FEATURE_COLS,
        save_path=out_csv
    )


Found test files: ['./data_filtering/filtered/TEST_00.csv', './data_filtering/filtered/TEST_01.csv', './data_filtering/filtered/TEST_02.csv', './data_filtering/filtered/TEST_03.csv', './data_filtering/filtered/TEST_04.csv', './data_filtering/filtered/TEST_05.csv', './data_filtering/filtered/TEST_06.csv', './data_filtering/filtered/TEST_07.csv', './data_filtering/filtered/TEST_08.csv', './data_filtering/filtered/TEST_09.csv']


/tmp/ipykernel_338269/2692946801.py:45: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ck, map_location=device)


[Saved] ./submission_TEST_00.csv | shape=(7, 193)


/tmp/ipykernel_338269/2692946801.py:45: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ck, map_location=device)


[Saved] ./submission_TEST_01.csv | shape=(7, 193)


...

[Saved] ./submission_TEST_07.csv | shape=(7, 193)


/tmp/ipykernel_338269/2692946801.py:45: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ck, map_location=device)


[Saved] ./submission_TEST_08.csv | shape=(7, 193)


/tmp/ipykernel_338269/2692946801.py:45: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ck, map_location=device)


[Saved] ./submission_TEST_09.csv | shape=(7, 193)


In [64]:
import pandas as pd
import glob

# submission_TEST_00.csv ~ submission_TEST_09.csv 파일을 번호 순서대로 불러와서 하나의 데이터프레임으로 합치기
submission_files = sorted(glob.glob("./submission_TEST_0*.csv"), key=lambda x: int(x.split("_")[-1].split(".")[0]))
dfs = []
for file in submission_files:
    df = pd.read_csv(file, index_col=0)
    dfs.append(df)
merged_submission = pd.concat(dfs, axis=0)

merged_submission.to_csv("merged_submission.csv")
print("merged_submission.csv 파일로 저장 완료")



merged_submission.csv 파일로 저장 완료
